- This notebook is a wip of the model engineering task, fine-tuning transformer model such as BERT and RoBERTa.
- Next Steps: 
    - Model optimisation for inference (reducing latency)
    - Per-entity evaluation
    - Error analysis
    - Best model export

Code source -> [article](https://medium.com/@whyamit101/fine-tuning-bert-for-named-entity-recognition-ner-b42bcf55b51d)

In [ ]:
# Uncomment in Colab / fresh environment
# !pip -q install datasets transformers seqeval scikit-learn pandas accelerate>=1.1.0

In [ ]:
import transformers
import accelerate
import torch

print(transformers.__version__)
print(accelerate.__version__)
print(torch.__version__)

### **IMPORT DEPENDENCIES**

In [ ]:
import os
import csv
import time
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from collections import Counter

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

### **1. CONFIG**

In [ ]:
# ==================== MODEL ====================
MODEL_NAME = "bert-base-uncased"
# MODEL_NAME = "roberta-base"           # Uncomment this line for RoBERTa fine-tuning

# ==================== PATH ====================
OUT_DIR = f"../results/checkpoints/{MODEL_NAME}"
LOG_DIR = f"../results/logs/{MODEL_NAME}"
RESULT_DIR = "../results/experiment_results.csv"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

# ==================== MODEL HYPERPARAMETERS ====================
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
EPOCH = 4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

# ==================== CSV Auto Logger ====================
def log_results_to_csv(file_path, data):
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode-"a", newline="") as file:
        writer = csv.DictWriter(file, filenames=data.keys())

        if not file_exists:
            writer.writeheader()

        writer.writenow(data)

### **2. LOAD DATASET**

In [ ]:
ds = load_from_disk("../data/processed/resume_ner_hf")

# ==================== Label Setup ====================
train_labels = [label for sample in ds["train"]["labels"] for label in sample]
label_list = sorted(set(train_labels))

id2label = {i: str(label) for i, label in enumerate(label_list)}
label2id = {str(label): i for i, label in enumerate(label_list)}

print(ds)
print("Labels:", id2label)

### **3. MODELLING**

In [ ]:
# ==================== Load Model ====================
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)           # for padding purposes

# ==================== Data Collator ====================
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)      # dataset is tokenized

# ==================== Metrics ====================
last_entity_report = None

def compute_metrics(pred):
    global last_entity_report

    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=2)

    true_labels = [
        [id2label[l] for l, p in zip(label, pred) if l != -100]
        for label, pred in zip(labels, preds)
    ]

    true_preds = [
        [id2label[p] for l, p in zip(label, pred) if l != -100]
        for label, pred in zip(labels, preds)
    ]

    precision = precision_score(true_labels, true_preds)
    recall = recall_score(true_labels, true_preds)
    f1 = f1_score(true_labels, true_preds)

    last_entity_report = classification_report(
        true_labels,
        true_preds,
        output_dict=True,
        zero_division=0
    )

    return {"precision": precision, "recall": recall, "f1": f1}

### **4. TRAINING PIPELINE**

In [ ]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCH,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_RATIO,

    #logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_dir=LOG_DIR,
    logging_steps=50,

    #fp16=torch.cuda.is_available(),
    seed=42
)

# ==================== Trainer ====================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    #tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ==================== Train & Time Tracking ====================
start_time = time.time()

trainer.train()

end_time = time.time()
training_time = end_time - start_time

print(f"\nTraining Time: {training_time/60:.2f} minutes ({training_time:.2f} seconds)")

### **5. TESTING**

In [ ]:
results = trainer.evaluate(ds["test"])
print("\nTest Results:", results)

In [ ]:
if device == "cuda":
    print(f"\nFinal GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Max GPU Memory Allocated: {torch.cuda.max_memory_allocated(0) / 1e9:.2f} GB")

In [ ]:
if device == "cuda":
    max_gpu_mem = torch.cuda.max_memory_allocated(0) / 1e9
else:
    max_gpu_mem = 0

log_data = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model":MODEL_NAME,
    "precision": results.get("eval_precision"),
    "recall": results.get("eval_recall"),
    "f1": results.get("eval_f1"),
    "training_time_min": round(training_time / 60, 2),
    "max_gpu_memory_gb": round(max_gpu_mem, 2),
    "device": device
}

log_file = RESULT_DIR
df = pd.DataFrame([log_data])

if os.path.exists(log_file):
    df.to_csv(log_file, mode="a", header=False, index=False)
else:
    df.to_csv(log_file, index=False)

print(f"\nResults logged to {log_file}")

In [ ]:
# ==================== Per-Entity Logging ====================
entity_rows = []

for entity, scores in last_entity_report.items():
    if isinstance(scores, dict) and "f1-score" in scores:
        entity_rows.append(
            {
                "model": MODEL_NAME,
                "entity": entity,
                "precision": scores["precision"],
                "recall": scores["recall"],
                "f1": scores["f1-score"]
            }
        )

entity_df = pd.DataFrame(entity_rows)
entity_df.to_csv("./results/experiment_results.csv", index=False)

entity_df